# 9a. Tuning to the junction counts: mid-block loading

**Question:** which lever brings the model's junction volumes to the JICA 2011-12 counts?
**Target:** GEH < 5 at all 6 tuning junctions (Chabahil, Gausala, Kalimati, New Baneshwor, Old Baneshwor, Thapathali), on the mean volume over 5 seeds, with road speeds still near the JICA speed survey (Ring Road 24.0, inner main roads 15.3 km/h).

The other 3 counted junctions (Balaju, Kalanki, Koteshwor) were set aside by a fixed random draw before any tuning run and are not shown here; they are checked only once a configuration has been chosen.

**This notebook:** Each zone loaded its trips at its highest-degree nodes, which are the big junctions themselves: about 1,860 PCU/h started or ended at each Baneshwor junction node, never passing its approach roads or its count. Here trips load at mid-block nodes (on one road, at least 50 m from any junction and 100 m from a counted one) nearest each zone's centre, with outer zones on main roads, across the best capacity and demand settings so far. Every configuration runs on seeds 1-5, next to the unchanged baseline. Volumes are measured in the second hour of a two-hour steady peak.

**Junction measure:** OpenStreetMap draws some large junctions as several nodes, so each junction is also scored as all traffic entering the nodes within 100 m of it (fixed before rescoring any run). Both measures are printed.

In [ ]:
import glob, os, shutil, subprocess, sys, zipfile

def find_project(marker):
    """Copy the project tree that contains `marker` from /kaggle/input to /kaggle/working/sim."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{marker}", recursive=True),
                  key=lambda h: ("build-demand" not in h, len(h)))   # prefer step 1's copy
    if not hits:                                        # dataset delivered as a zip
        for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
            zipfile.ZipFile(z).extractall("/kaggle/working/_unzipped")
        hits = sorted(glob.glob(f"/kaggle/working/_unzipped/**/{marker}", recursive=True), key=len)
    src = hits[0][: -len(marker)].rstrip("/")
    shutil.copytree(src, "/kaggle/working/sim", dirs_exist_ok=True)
    os.chdir("/kaggle/working/sim")
    print("project from", src)

def run(*cmd, tries=1):
    """Run a script, streaming its output; retry resumable drivers if a worker is killed."""
    for attempt in range(tries):
        p = subprocess.Popen([sys.executable, "-u", *cmd], stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end="")
        if p.wait() == 0:
            return
        print(f"exit code {p.returncode}, attempt {attempt + 1} of {tries}")
    raise RuntimeError(" ".join(cmd))

find_project("scripts/tune_counts.py")

In [ ]:
!pip install -q uxsim==1.14.2

In [ ]:
run("scripts/build_minor_ways.py")
run("scripts/build_od_jica.py", "--data", "data/jica_net")

In [ ]:
run("scripts/tune_counts.py", "--idea", "midblock", "--seeds", "5", "--jobs", "3", tries=3)

In [ ]:
run("scripts/tune_counts.py", "--idea", "midblock", "--summary-only", "--radius", "100")